# Run only the missing experiments (Google Colab)

The 170 runs already in `results/runs.csv` are **not** repeated: this notebook clones the repo (which includes
`runs.csv`), and every script skips runs that are already logged. What is still missing:

| Step | What | Runs |
|---|---|---|
| 1 | Gemma 3 4B zero-shot | 1 |
| 2 | LoRA fine-tuning of the web-trained DINOv3 ViT-L, 5 / 50 / 500 labels per class (about 1 hour on a T4) | 9 |

Turn either step off in the next cell if you do not need it.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. Click the key icon (Secrets) in the left sidebar, add `HF_TOKEN` = your Hugging Face read token, and turn on *Notebook access*.
3. Accept the licence on [google/gemma-3-4b-it](https://huggingface.co/google/gemma-3-4b-it) (step 1) and
   [timm/vit_large_patch16_dinov3.lvd1689m](https://huggingface.co/timm/vit_large_patch16_dinov3.lvd1689m) (step 2).
4. *Runtime → Run all*. Allow Google Drive access when asked: results are copied to Drive after every step,
   so if Colab disconnects you can just *Run all* again and it continues where it stopped.

At the end a `results.zip` is downloaded. Unzip it into your local repo, overwrite, commit and push.

In [ ]:
RUN_GEMMA = True
RUN_DINO_WEB_LORA = True
SAVE_TO_DRIVE = True

In [ ]:
import os, shutil, subprocess
DRIVE_DIR = "/content/drive/MyDrive/eurosat-fm-results"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

os.chdir("/content")
if not os.path.exists("eurosat-foundation-models"):
    subprocess.run(["git", "clone", "https://github.com/kingmars2022/eurosat-foundation-models.git"], check=True)
os.chdir("/content/eurosat-foundation-models")

if SAVE_TO_DRIVE and os.path.exists(f"{DRIVE_DIR}/runs.csv"):
    shutil.copytree(DRIVE_DIR, "results", dirs_exist_ok=True)
    print("Restored earlier progress from Google Drive")

def save_progress():
    if SAVE_TO_DRIVE:
        shutil.copytree("results", DRIVE_DIR, dirs_exist_ok=True)
        print("Progress saved to", DRIVE_DIR)

def count_runs():
    import pandas as pd
    df = pd.read_csv("results/runs.csv")
    print(len(df), "runs logged;", df.groupby("method").size().to_dict())

count_runs()

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF token loaded")
except Exception as e:
    print("No HF_TOKEN secret found - DINOv3 and Gemma downloads will fail:", e)

In [ ]:
!python -m eurofm.prepare_data --source torchvision --root data/raw

## Step 1 - Gemma 3 4B zero-shot

In [ ]:
if RUN_GEMMA:
    !python -m eurofm.vlm eval --model gemma3-4b
    save_progress()
    count_runs()

## Step 2 - LoRA fine-tuning of the web-trained DINOv3 ViT-L

In [ ]:
if RUN_DINO_WEB_LORA:
    !python -m eurofm.finetune --mode lora --encoders dinov3_vitl16_web --budgets 5,50,500 --seeds 0 1 2
    save_progress()
    count_runs()

## Tables, figures and download

In [ ]:
!python -m eurofm.report
save_progress()
from IPython.display import Markdown, Image, display
display(Markdown(open("results/summary.md").read()))
display(Image("results/figures/learning_curves.png"))

In [ ]:
!rm -f /content/results.zip && zip -qr /content/results.zip results
from google.colab import files
files.download("/content/results.zip")